# 17 — Revision Not Reset

## When does pruning behave like revision?

This notebook extends the pruning-RML vocabulary.

Question:

> When does inherited structure survive compression and retraining?

Outputs:

```text
results/17_revision_not_reset.csv
figures/17_revision_not_reset.png
```

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git","clone",REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print(repo)

In [ ]:
pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg/"__init__.py").write_text('__version__="0.1.0"\n', encoding="utf-8")

(pkg/"paths.py").write_text('''from pathlib import Path
def ensure_dirs(root, names=("results","figures","reports")):
    root = Path(root)
    out={}
    for n in names:
        p=root/n
        p.mkdir(parents=True, exist_ok=True)
        out[n]=p
    return out
''', encoding="utf-8")

(pkg/"revision.py").write_text('''def revision_score(preserve,relearn):
    return preserve - relearn

def preservation_score(v):
    return v

def relearning_score(v):
    return v
''', encoding="utf-8")
print("helpers ready")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

## Revision schema

Reset rebuilds.

Revision preserves and modifies.

In [ ]:
rows = [
    ["random initialization","none","no","high",0],
    ["layer removal","partial","some","medium",1],
    ["width pruning","moderate","yes","medium",2],
    ["structured pruning","moderate","yes","low",3],
    ["sparse pruning","high","yes","low",4],
    ["prune + retrain","high","yes","low",5],
]

df = pd.DataFrame(
    rows,
    columns=[
        "operation",
        "starting_structure",
        "preserves_information",
        "requires_relearning",
        "revision_score",
    ],
)

df

In [ ]:
csv_path = results / "17_revision_not_reset.csv"
df.to_csv(csv_path, index=False)

csv_path

In [ ]:
fig_path = figures / "17_revision_not_reset.png"

ax = df.plot.bar(
    x="operation",
    y="revision_score",
    legend=False,
    figsize=(9,5),
)

ax.set_title("Revision Not Reset")
ax.set_ylabel("Revision score")

plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print(fig_path)

## Result

Notebook 17 asks whether inherited structure survives compression.

Next:

```text
23_surviving_residues.ipynb
```